# 📝 English Handwritten Examination OCR & Multi-Backend Benchmark
### Thesis Research: Stage 1 Single-Pass OCR & Confidence Calibration
This notebook is optimized for **Kaggle (Free GPU - Tesla T4 / P100)** to test, benchmark, and compare:
1. **Infinite-OCR / Vision-Language Model** (`nanonets/Nanonets-OCR2-3B` or `Qwen2.5-VL-3B-Instruct`)
2. **TrOCR** (`microsoft/trocr-base-handwritten`)
3. **EasyOCR** (CRAFT + BiLSTM baseline)


In [ ]:
# Step 1: Install required packages in Kaggle environment
!pip install -q transformers accelerate qwen-vl-utils easyocr pydantic torch pillow torchvision pandas


In [ ]:
# Step 2: Verify GPU Accelerator
import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), 'GB')
else:
    print('Running on CPU. Tip: Enable GPU via Notebook Settings -> Accelerator -> GPU T4 x2')


In [ ]:
# Step 3: Clone or Load the Repository Code
import os, sys
if not os.path.exists('src'):
    !git clone -b siam https://github.com/mdsiam9646/Ugrad-Thesis-Script-Checking-With-Multimodal-AI.git repo
    os.chdir('repo')
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Workspace root:', os.getcwd())


In [ ]:
# Step 4: Preview Input Image
from PIL import Image
import matplotlib.pyplot as plt

image_path = 'data/samples/sample.png'
if os.path.exists(image_path):
    img = Image.open(image_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title(f'Sample: {image_path} ({img.width}x{img.height})')
    plt.show()
else:
    print(f'Please upload an image to {image_path}')


In [ ]:
# Step 5: Run Infinite-OCR (Vision-Language Model)
!python scripts/run_ocr.py --input data/samples/sample.png --backend infinite_ocr


In [ ]:
# Step 6: Run TrOCR (microsoft/trocr-base-handwritten)
!python scripts/run_ocr.py --input data/samples/sample.png --backend trocr


In [ ]:
# Step 7: Run EasyOCR Baseline (CRAFT + BiLSTM)
!python scripts/run_ocr.py --input data/samples/sample.png --backend easyocr


In [ ]:
# Step 8: Side-by-Side OCR Comparison Table
import json, glob
import pandas as pd
from IPython.display import display, HTML

records = []
for json_file in glob.glob('outputs/ocr/*.json'):
    with open(json_file, 'r', encoding='utf-8') as f:
        d = json.load(f)
        records.append({
            'Backend': d.get('ocr', {}).get('backend'),
            'Model': d.get('ocr', {}).get('model'),
            'Confidence': d.get('ocr', {}).get('confidence'),
            'Time (s)': d.get('metadata', {}).get('processing_time_seconds'),
            'Device': d.get('metadata', {}).get('device'),
            'Transcribed Text': d.get('ocr', {}).get('normalized_text')
        })

if records:
    df = pd.DataFrame(records)
    display(HTML(df.to_html(index=False)))
